In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
from matplotlib.container import BarContainer

import utils

plt.style.use('default')

In [ ]:
CI = 95  # 95 % CI
ALPHA = 1 - CI / 100.0
TYPES = ['INITIAL', 'COVERAGE', 'MUTATION']
TOKEN_METRICS = ['INPUT_TOKEN_COUNT', 'OUTPUT_TOKEN_COUNT']

In [ ]:
df = utils.union_read_csv('data/Gson_iteration-token-costs.csv', 'data/Lang_iteration-token-costs.csv', 'data/JacksonCore_iteration-token-costs.csv')

agg = {}

for m in TOKEN_METRICS:
    agg[m] = utils.aggregate_for_metric(df, ['FEEDBACK_TYPE'], lambda r: r[m], ALPHA)

data = pd.concat(agg, names=['metric']).reset_index(level=0)
data['FEEDBACK_TYPE'] = pd.Categorical(data['FEEDBACK_TYPE'], categories=TYPES, ordered=True)
pivot = data.pivot(index='metric', columns='FEEDBACK_TYPE')

fig, axes = plt.subplots(figsize=utils.A5_LANDSCAPE_INCHES, dpi=300)

pivot['mean'].plot(kind='bar', yerr=pivot['hw'], ax=axes)

for cont in axes.containers:
    if isinstance(cont, BarContainer):
        axes.bar_label(cont, padding=2, fontsize=9, fmt=lambda v: f'{v:,.0f}'.replace(',', ' '))

axes.set_xlabel('')
axes.set_ylabel('Tokens')
axes.tick_params(axis='x', labelsize=8, rotation=0)
axes.yaxis.set_major_formatter(ticker.FuncFormatter(lambda val, pos: f'{val:,.0f}'.replace(',', ' ')))
axes.grid(linestyle='--', linewidth=.5, alpha=0.5, axis='y')

plt.show()